# Session 12: Text-RAG Pipeline — Build Your First RAG System

**Welcome!** In this session, you'll build a complete Retrieval-Augmented Generation (RAG) pipeline from scratch.

- **Duration:** 3 hours
- **Level:** Beginner
- **What you'll learn:** Chunking, retrieval, generation, routing, and evaluation
- **What you'll build:** A Q&A chatbot powered by your own documents

In [ ]:
# Setup: Install packages and import libraries
# !pip install langchain-core langchain-google-genai langchain-community langchain-text-splitters chromadb -q

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os

# Initialize LLM and embeddings
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

print("✓ LLM and embeddings initialized")

## What is RAG?

**RAG = Retrieval-Augmented Generation**

Large language models (LLMs) are trained on data from the past. They don't know about:  
- Your private documents  
- Recent events after training cutoff  
- Company-specific knowledge  

**RAG solves this:** Retrieve relevant documents first, then let the LLM answer based on those documents.

### The RAG Pipeline

```
┌─────────────┐
│  Documents  │
└──────┬──────┘
       │
       ├─→ Chunk ─→ Embed ─→ Store in Vector DB (Chroma)
       │                              │
       │                              ↓
┌──────────────┐     ┌──────────────────────┐
│   Question   │────→│ Embed & Search Vector│
└──────────────┘     └──────────────────────┘
       │                              │
       │                              ↓
       └─→ Retrieve Top-K Chunks ─→ LLM ─→ Answer
```

## Part 2: Loading & Chunking Documents

**Step 1: Create Sample Documents**

In [ ]:
# Sample documents about Indian tourism
sample_text = """
India Tourism Guide

Goa is famous for its beautiful beaches and laid-back atmosphere. Visitors can enjoy water sports, 
explore colonial architecture, and relax on pristine sandy shores. The beaches of Goa are especially 
popular during the winter months (November to February) when the weather is perfect for swimming and 
sunbathing. Arambol, Baga, and Calangute beaches are among the most visited.

Kerala, located on the southern coast of India, is known for its backwaters, houseboat rides, and 
lush green landscapes. The state is called 'God's Own Country' because of its natural beauty. Visitors 
can experience Ayurvedic treatments, enjoy fresh seafood, and travel through serene waterways lined 
with coconut palms and mangroves.

Varanasi is one of the holiest cities in India, situated on the banks of the Ganges River. It is a 
major pilgrimage site for Hindus who believe bathing in the Ganges purifies the soul. The city is 
famous for its ancient temples, ghats (steps leading to the river), and spiritual atmosphere. Sunrise 
boat rides on the Ganges are a must-see experience.

Rajasthan is the land of maharajas (kings) and is known for its magnificent palaces, forts, and desert 
landscapes. The state is home to the famous Taj Mahal in Agra, one of the Seven Wonders of the World. 
Visitors can also explore Jaipur's Pink City, ride camels in the Thar Desert, and visit historic forts 
like Mehrangarh and Amber Fort.

Himachal Pradesh in the north offers adventure and mountain beauty. This state is perfect for hiking, 
paragliding, and mountaineering. Towns like Shimla, Manali, and Dharamshala are popular tourist destinations. 
Visitors can trek through alpine meadows, visit Buddhist monasteries, and enjoy panoramic views of the 
Himalayan peaks.

Food in India varies greatly by region. South India is famous for idli, dosa, and sambar. North India 
loves tandoori chicken, biryani, and naan bread. Street food like chaat and samosas are popular everywhere. 
Indian food is characterized by the use of spices like turmeric, cumin, coriander, and chili.
"""

print(f"Total characters in sample text: {len(sample_text)}")
print(f"Sample length: {len(sample_text)} characters")

## Step 2: Chunking — Why and How

**Why chunk?**
- Vector databases and LLMs have token/context limits
- Smaller chunks are easier to search and retrieve
- Better matching: a query is more likely to match a small relevant chunk

**How to chunk?**
- `chunk_size`: How many characters per chunk (e.g., 500)
- `chunk_overlap`: How much text to repeat between chunks (e.g., 50)
  - Overlap prevents losing information at chunk boundaries

In [ ]:
# RecursiveCharacterTextSplitter: splits on separators like \n\n, \n, or space
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(sample_text)

print(f"Split into {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} characters):")
    print(chunk[:100] + "...")  # Show first 100 chars
    print()

In [ ]:
# Experiment: Compare different chunk sizes
print("=== Chunking Strategy Comparison ===")
print()

for size in [256, 512, 1024]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    c = s.split_text(sample_text)
    avg_len = sum(len(x) for x in c) // len(c) if c else 0
    print(f"Chunk size {size}: {len(c)} chunks, avg {avg_len} chars/chunk")

## Part 3: Embed & Store in Vector Database

**Step 3: Store chunks in Chroma**

We'll create embeddings for each chunk and store them in Chroma (an in-memory vector database).

In [ ]:
# Create Chroma vectorstore from chunks
# Chroma automatically embeds each chunk using our embeddings model
vectorstore = Chroma.from_texts(chunks, embeddings)

num_stored = vectorstore._collection.count()
print(f"✓ Stored {num_stored} chunks in Chroma")

## Part 4: The Retriever

**Step 4: Create a retriever**

A retriever searches the vector store for the top-k most similar documents to a query.

In [ ]:
# Create a retriever that returns top-3 documents
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test the retriever
results = retriever.invoke("best beaches in India")
print(f"Retrieved {len(results)} documents:\n")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content[:150] + "...")  # Show first 150 chars
    print()

In [ ]:
# Try another query
results = retriever.invoke("food and cuisine")
print(f"Retrieved {len(results)} documents for 'food and cuisine':\n")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content[:150] + "...")  # Show first 150 chars
    print()

## Part 5: The RAG Chain

**Step 5: Build the RAG chain**

Connect the retriever to the LLM using LangChain Expression Language (LCEL).

In [ ]:
# Define the RAG prompt template
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say 'I don't have information about that.'

Context:
{context}

Question: {question}

Answer:"""
)

In [ ]:
# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain with LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG chain created")

In [ ]:
# Test the RAG chain with a simple query
answer = rag_chain.invoke("What are the best beaches in Goa?")
print("Question: What are the best beaches in Goa?")
print(f"Answer: {answer}")

In [ ]:
# Test with multiple queries, including one outside our documents
queries = [
    "What food is popular in Kerala?",
    "Tell me about temples in Varanasi",
    "What is the capital of France?",  # Not in our documents!
]

for q in queries:
    print(f"Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"A: {answer}\n")

## Part 6: The Router Pattern

**Not every query needs retrieval!**

A **router** classifies whether a query needs document retrieval or can be answered with general knowledge.

- `retrieval` queries: "What beaches are in Goa?" (needs our documents)
- `general` queries: "What is 2+2?" (general knowledge)

In [ ]:
# Simple router using the LLM
def route_query(question):
    # Ask the LLM to classify the question
    classification = llm.invoke(
        f"""Classify this question. Reply with ONLY 'retrieval' or 'general'.\n
'retrieval' = needs specific document knowledge\n
'general' = general knowledge question\n\n
Question: {question}\nClassification:"""
    )
    
    route = classification.content.strip().lower()
    print(f"[Route: {route}]", end=" ")
    
    if "retrieval" in route:
        return rag_chain.invoke(question)
    else:
        return llm.invoke(question).content


In [ ]:
# Test the router
print("=== Router Test ===")
print()

print("Q: What beaches are in Goa?")
answer1 = route_query("What beaches are in Goa?")
print(f"A: {answer1}\n")

print("Q: What is 2 + 2?")
answer2 = route_query("What is 2 + 2?")
print(f"A: {answer2}")

## Part 7: Evaluating Your RAG

**How do you know if your RAG is working?**

We use evaluation metrics that compare retrieved documents to ground truth (correct answers).

### Metrics

- **Precision@k**: Of the top-k documents retrieved, how many are correct?
  - Formula: `correct_docs_in_top_k / k`

- **Recall@k**: Of all the correct documents, how many appear in top-k?
  - Formula: `correct_docs_in_top_k / total_correct_docs`

- **F1@k**: Harmonic mean of Precision and Recall
  - Formula: `2 * (Precision * Recall) / (Precision + Recall)`

In [ ]:
# Simple evaluation metrics
def precision_at_k(retrieved, relevant, k):
    """Precision@k: fraction of top-k docs that are relevant"""
    retrieved_k = retrieved[:k]
    relevant_found = sum(1 for doc in retrieved_k if doc in relevant)
    return relevant_found / k

def recall_at_k(retrieved, relevant, k):
    """Recall@k: fraction of relevant docs that appear in top-k"""
    retrieved_k = retrieved[:k]
    relevant_found = sum(1 for doc in retrieved_k if doc in relevant)
    return relevant_found / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    """F1@k: harmonic mean of Precision and Recall"""
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

In [ ]:
# Example: Evaluate a retrieval scenario
# Suppose we retrieved these chunks (in order)
retrieved_chunks = ["chunk_1", "chunk_3", "chunk_5", "chunk_2", "chunk_7"]

# And we know these chunks are actually relevant
relevant_chunks = ["chunk_1", "chunk_3", "chunk_4"]  # Ground truth

print("=== Evaluation Results ===")
print(f"Retrieved: {retrieved_chunks}")
print(f"Relevant:  {relevant_chunks}")
print()

for k in [1, 3, 5]:
    p = precision_at_k(retrieved_chunks, relevant_chunks, k)
    r = recall_at_k(retrieved_chunks, relevant_chunks, k)
    f1 = f1_at_k(retrieved_chunks, relevant_chunks, k)
    print(f"k={k}: Precision={p:.2f}, Recall={r:.2f}, F1={f1:.2f}")

### Interpreting Results

- **High Precision, Low Recall**: You're being too conservative (missing some relevant docs)
- **Low Precision, High Recall**: You're retrieving too much (including irrelevant docs)
- **High F1**: Good balance between precision and recall

### For Production RAG Systems

The [**Ragas framework**](https://github.com/explodinggradients/ragas) provides automated evaluation with metrics like:
- **Context Precision**: Are retrieved chunks relevant?
- **Context Recall**: Did you retrieve all relevant chunks?
- **Faithfulness**: Is the answer faithful to the context?
- **Answer Relevance**: Does the answer answer the question?

## Exercise 1: Build a FAQ Bot

**Task:** Create a FAQ bot that stores and retrieves Q&A pairs.

Steps:
1. Create a sample FAQ dataset (Q&A pairs about Indian tourism)
2. Store the answers in a vector database
3. When a user asks a question, retrieve the most similar Q&A pair
4. Return the retrieved answer (or generate a new one using RAG)

Hints:
- Store both questions and answers as documents
- Use the retriever to find similar questions
- Compare user query similarity to stored questions

In [ ]:
# Exercise 1: FAQ Bot
# TODO: Implement the FAQ bot below

# Step 1: Create sample FAQ
faq_data = {
    "What is the best time to visit Goa?": "...",  # FILL IN
    "What food should I try in Kerala?": "...",  # FILL IN
    # Add more Q&A pairs
}

# Step 2: Create retriever for FAQ
# faq_texts = ... # FILL IN
# faq_vectorstore = Chroma.from_texts(...)
# faq_retriever = faq_vectorstore.as_retriever(...)

# Step 3: Test the FAQ bot
# user_question = "When should I go to Goa?"
# similar_qa = ... # FILL IN
# print(similar_qa)

## Exercise 2: Experiment with Chunk Sizes

**Task:** Evaluate how different chunk sizes affect retrieval quality.

Steps:
1. Create vectorstores with different chunk sizes (256, 512, 1024, 2048)
2. Test the same query on each vectorstore
3. Compare the retrieved results (relevance, specificity, coverage)
4. Which chunk size works best for your use case?

Questions to consider:
- Does a smaller chunk size retrieve more specific information?
- Does a larger chunk size provide better context?
- Is there a trade-off between specificity and context?

In [ ]:
# Exercise 2: Chunk Size Experiment
# TODO: Implement the experiment below

chunk_sizes = [256, 512, 1024, 2048]
test_query = "best beaches"

results = {}
for size in chunk_sizes:
    # Step 1: Create splitter with this chunk size
    # splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    
    # Step 2: Split documents
    # chunks = ... # FILL IN
    
    # Step 3: Create vectorstore and retriever
    # vs = ... # FILL IN
    # ret = ... # FILL IN
    
    # Step 4: Retrieve and store results
    # results[size] = ret.invoke(...)

# Step 5: Compare results
# for size, docs in results.items():
#     print(f"Chunk size {size}: {len(docs)} docs retrieved")

## Summary

Congratulations! You've built a complete RAG pipeline from scratch. You learned:

✓ **Chunking**: Breaking documents into manageable pieces  
✓ **Retrieval**: Finding relevant documents using semantic search  
✓ **Generation**: Using an LLM to answer based on retrieved context  
✓ **Routing**: Determining when to use retrieval vs. general knowledge  
✓ **Evaluation**: Measuring RAG quality with metrics  

## What's Next?

**Session 13: Multimodal RAG** — Extend RAG to work with images, tables, and PDFs!

## Resources

- [LangChain Documentation](https://python.langchain.com/)
- [Chroma Vector Database](https://www.trychroma.com/)
- [Ragas Evaluation Framework](https://github.com/explodinggradients/ragas)
- [Gemini API Documentation](https://ai.google.dev/)